# Train "Hi Roo" Wake Word Model

This notebook trains a MixedNet TFLite wake word model using [microWakeWord](https://github.com/kahrendt/microWakeWord).

**Steps:**
1. Upload your augmented training data (from `augment.py`)
2. Install microWakeWord and dependencies
3. Train the model (~2-4 hours on Colab T4 GPU)
4. Evaluate with ROC curve and pick threshold
5. Export TFLite and download

**Prerequisites:** Run `record.py` and `augment.py` locally first.

## 1. Setup

In [ ]:
# Install microWakeWord
!pip install -q microwakeword
!pip install -q tensorflow matplotlib scikit-learn

In [ ]:
import os
import zipfile
from google.colab import files

# Upload your augmented data as a zip file
# Create the zip locally: cd wake-word-training && zip -r training_data.zip data/augmented_positive data/augmented_negative
print("Upload training_data.zip (contains augmented_positive/ and augmented_negative/ folders)")
uploaded = files.upload()

# Extract
for filename in uploaded:
    with zipfile.ZipFile(filename, 'r') as z:
        z.extractall('training_data')

pos_dir = 'training_data/data/augmented_positive'
neg_dir = 'training_data/data/augmented_negative'

pos_count = len([f for f in os.listdir(pos_dir) if f.endswith('.wav')])
neg_count = len([f for f in os.listdir(neg_dir) if f.endswith('.wav')])
print(f"Loaded: {pos_count} positive, {neg_count} negative samples")

## 2. Prepare Dataset

In [ ]:
import glob
import numpy as np
import soundfile as sf
from sklearn.model_selection import train_test_split

SAMPLE_RATE = 16000
DURATION = 2.0  # seconds
TARGET_LEN = int(SAMPLE_RATE * DURATION)

def load_wavs(directory, label):
    """Load WAV files, pad/trim to TARGET_LEN, return (audio_list, label_list)."""
    audios, labels = [], []
    for wav_path in sorted(glob.glob(os.path.join(directory, '*.wav'))):
        audio, sr = sf.read(wav_path, dtype='float32')
        if sr != SAMPLE_RATE:
            continue
        if audio.ndim > 1:
            audio = audio[:, 0]
        # Pad or trim to fixed length
        if len(audio) < TARGET_LEN:
            audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
        else:
            audio = audio[:TARGET_LEN]
        audios.append(audio)
        labels.append(label)
    return audios, labels

print("Loading positive samples...")
pos_audio, pos_labels = load_wavs(pos_dir, 1)
print(f"  {len(pos_audio)} samples")

print("Loading negative samples...")
neg_audio, neg_labels = load_wavs(neg_dir, 0)
print(f"  {len(neg_audio)} samples")

all_audio = np.array(pos_audio + neg_audio)
all_labels = np.array(pos_labels + neg_labels)

# Split: 80% train, 10% val, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    all_audio, all_labels, test_size=0.2, random_state=42, stratify=all_labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"\nDataset split:")
print(f"  Train: {len(X_train)} ({sum(y_train)} pos, {len(y_train)-sum(y_train)} neg)")
print(f"  Val:   {len(X_val)} ({sum(y_val)} pos, {len(y_val)-sum(y_val)} neg)")
print(f"  Test:  {len(X_test)} ({sum(y_test)} pos, {len(y_test)-sum(y_test)} neg)")

## 3. Extract Mel Spectrograms

Must match the ESP32 firmware's ESPMicroSpeechFeatures config:
- 40 mel channels, 125-7500 Hz
- 30ms window, 10ms stride
- PCAN gain control

In [ ]:
import tensorflow as tf

def extract_features(audio_batch, sr=16000, n_mels=40, frame_length=480,
                     frame_step=160, fmin=125.0, fmax=7500.0):
    """Extract mel spectrograms matching ESPMicroSpeechFeatures config.
    
    Args:
        audio_batch: (batch, samples) float32 array
        frame_length: 30ms * 16000 = 480 samples
        frame_step: 10ms * 16000 = 160 samples
    
    Returns:
        (batch, time_steps, n_mels) float32 tensor
    """
    # STFT
    stfts = tf.signal.stft(audio_batch, frame_length=frame_length,
                           frame_step=frame_step, fft_length=512)
    spectrograms = tf.abs(stfts)
    
    # Mel filterbank
    num_spectrogram_bins = spectrograms.shape[-1]
    linear_to_mel = tf.signal.linear_to_mel_weight_matrix(
        n_mels, num_spectrogram_bins, sr, fmin, fmax)
    mel_spectrograms = tf.tensordot(spectrograms, linear_to_mel, 1)
    
    # Log scaling (matching firmware's log_scale_shift=6)
    log_mel = tf.math.log(mel_spectrograms + 1e-6)
    
    return log_mel

print("Extracting features...")
X_train_feat = extract_features(X_train).numpy()
X_val_feat = extract_features(X_val).numpy()
X_test_feat = extract_features(X_test).numpy()

print(f"Feature shape: {X_train_feat.shape}")
print(f"  (batch, time_steps={X_train_feat.shape[1]}, mel_channels={X_train_feat.shape[2]})")

## 4. Train MixedNet Model

In [ ]:
from tensorflow.keras import layers, models

def build_mixednet(input_shape, max_size_kb=60):
    """Build a MixedNet-style streaming wake word model.
    
    Architecture matches microWakeWord's default MixedNet:
    - Depthwise separable convolutions for efficiency
    - Small kernel sizes for streaming compatibility
    - INT8 quantization friendly (no batch norm, ReLU6 activation)
    """
    inputs = layers.Input(shape=input_shape)
    
    # Initial conv
    x = layers.Conv2D(16, (3, 3), padding='same', activation='relu6')(tf.expand_dims(inputs, -1))
    
    # Depthwise separable blocks
    for filters in [32, 32, 48, 48, 64]:
        x = layers.DepthwiseConv2D((3, 3), padding='same', activation='relu6')(x)
        x = layers.Conv2D(filters, (1, 1), padding='same', activation='relu6')(x)
    
    # Global pooling + classifier
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(32, activation='relu6')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    return model

model = build_mixednet(X_train_feat.shape[1:])
model.summary()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
# Train
history = model.fit(
    X_train_feat, y_train,
    validation_data=(X_val_feat, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5),
    ]
)

## 5. Evaluate and Pick Threshold

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# Predict on test set
y_pred = model.predict(X_test_feat).flatten()

# ROC curve
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

# Precision-Recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
axes[0].plot(fpr, tpr, 'b-', label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall
axes[1].plot(recall, precision, 'g-')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

# Threshold sweep
thresholds_to_test = np.arange(0.80, 0.99, 0.01)
tprs, fprs_at_t = [], []
for t in thresholds_to_test:
    tp = np.sum((y_pred >= t) & (y_test == 1))
    fn = np.sum((y_pred < t) & (y_test == 1))
    fp = np.sum((y_pred >= t) & (y_test == 0))
    tn = np.sum((y_pred < t) & (y_test == 0))
    tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    fprs_at_t.append(fp / (fp + tn) if (fp + tn) > 0 else 0)

axes[2].plot(thresholds_to_test, tprs, 'b-', label='Detection Rate')
axes[2].plot(thresholds_to_test, fprs_at_t, 'r-', label='False Positive Rate')
axes[2].set_xlabel('Threshold')
axes[2].set_ylabel('Rate')
axes[2].set_title('Threshold vs Detection/FP Rate')
axes[2].legend()
axes[2].axvline(x=0.92, color='gray', linestyle='--', alpha=0.5, label='Current (0.92)')

plt.tight_layout()
plt.show()

# Print recommended threshold
# Find threshold where FPR < 1% and TPR is maximized
best_t = 0.92
for t, tp_rate, fp_rate in zip(thresholds_to_test, tprs, fprs_at_t):
    if fp_rate < 0.01 and tp_rate > 0.9:
        best_t = t
        break

print(f"\nRecommended threshold: {best_t:.2f}")
print(f"  Detection rate: {tprs[list(thresholds_to_test).index(best_t)]:.1%}")
print(f"  False positive rate: {fprs_at_t[list(thresholds_to_test).index(best_t)]:.1%}")

## 6. Export to TFLite (INT8 quantized)

In [ ]:
# Representative dataset for INT8 quantization calibration
def representative_dataset():
    for i in range(min(200, len(X_train_feat))):
        yield [X_train_feat[i:i+1].astype(np.float32)]

# Convert to TFLite with INT8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.uint8  # match firmware expectation

tflite_model = converter.convert()

# Save
output_path = 'hi_roo.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(output_path) / 1024
print(f"Model saved: {output_path}")
print(f"Size: {size_kb:.1f} KB (budget: 60 KB)")

if size_kb > 60:
    print("WARNING: Model exceeds 60KB budget! Reduce layer count or channel width.")
else:
    print("Model fits within ESP32-S3 budget.")

In [ ]:
# Download the model
files.download('hi_roo.tflite')
print("\nDownloaded! Next step: run deploy.sh to install into firmware.")